In [24]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# =====================================================================
# 1. DATA INGESTION & EXPLORATION
# =====================================================================
filename = 'churn-bigml-20.csv'

if not os.path.exists(filename):
    raise FileNotFoundError(
        f"Missing dataset file '{filename}' in current directory. "
        f"Please verify it is uploaded before running."
    )

print(f"⏳ Loading raw target dataset: '{filename}'...")
df = pd.read_csv(filename)

# Isolate features (X) and label targets (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].astype(int)  # Convert boolean True/False to binary 1/0

print(f"📊 Initial Raw State: {X.shape[0]} samples | {X.shape[1]} raw features.")

# =====================================================================
# 2. SEPARATING TRAINING AND TESTING MATRICES (Anti-Leakage Split)
# =====================================================================
# We split FIRST so that test metrics remain completely unseen by the scaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✔️ Split complete: Training rows = {len(X_train)} | Testing rows = {len(X_test)}")

# =====================================================================
# 3. IDENTIFYING FEATURE TYPES FOR AUTOMATED PIPELINES
# =====================================================================
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"🔍 Feature Type Mapping:")
print(f"   -> Numerical Columns   ({len(numerical_cols)}): {numerical_cols[:3]}... (etc)")
print(f"   -> Categorical Columns ({len(categorical_cols)}): {categorical_cols}")

# =====================================================================
# 4. BUILDING THE COLUMNTRANSFORMER PROCESSING ENGINE
# =====================================================================
print("\n⚙️ Constructing object-oriented pipeline transformers...")

# Numeric Pipeline: Patch missing data with the median, then scale to mean=0, std=1
numeric_transformer = ColumnTransformer(
    transformers=[
        ('impute_num', SimpleImputer(strategy='median'), numerical_cols)
    ],
    remainder='passthrough'
)

# Categorical Pipeline: Pass-through string missing data, then expand via One-Hot encoding
# We set handle_unknown='ignore' so unseen text patterns in deployment won't crash the architecture.
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Assemble everything into a unified matrix processor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols), # Applies standardization
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols) # Applies encoding
    ]
)

# =====================================================================
# 5. EXECUTING THE MATHEMATICAL TRANSFORMATION
# =====================================================================
print("🧮 Fitting pipeline weights on train features and transforming datasets...")

# Fit on training data ONLY to harvest means/variance, then transform both
X_train_clean = preprocessor.fit_transform(X_train)
X_test_clean = preprocessor.transform(X_test)

# Extract newly minted feature headers created during the one-hot expansion phase
encoded_cat_names = preprocessor.transformers_[1][1].get_feature_names_out(categorical_cols)
final_feature_names = numerical_cols + list(encoded_cat_names)

# Reassemble into clean dataframes for presentation inspection
X_train_final_df = pd.DataFrame(X_train_clean, columns=final_feature_names)

# =====================================================================
# 6. PRODUCTION QUALITY CHECK SCORECARD
# =====================================================================
print("\n" + "="*60)
print("             DATA PREPROCESSING QUALITY SCORECARD")
print("="*60)
print(f" -> Final Processed Train Shape     : {X_train_clean.shape}")
print(f" -> Final Processed Test Shape      : {X_test_clean.shape}")
print(f" -> Total Expanded Sparse Features  : {X_train_clean.shape[1]}")
print(f" -> Remaining Null Values Vector    : {np.isnan(X_train_clean).sum()}")
print(f" -> Feature Mean Range Verification : Close to {X_train_clean[:, 0].mean():.2f}")
print(f" -> Feature Std Dev Verification    : {X_train_clean[:, 0].std():.2f}")
print("-"*60)
print("\nFirst 3 Processed Samples Matrix Snippet (View of Standardized Columns):")
print(X_train_final_df.iloc[:3, :4].to_string())
print("="*60)

print("\n✔️ Complete! Your input data is now fully optimized and clean for any ML algorithm.")

⏳ Loading raw target dataset: 'churn-bigml-20.csv'...
📊 Initial Raw State: 667 samples | 19 raw features.
✔️ Split complete: Training rows = 533 | Testing rows = 134
🔍 Feature Type Mapping:
   -> Numerical Columns   (16): ['Account length', 'Area code', 'Number vmail messages']... (etc)
   -> Categorical Columns (3): ['State', 'International plan', 'Voice mail plan']

⚙️ Constructing object-oriented pipeline transformers...
🧮 Fitting pipeline weights on train features and transforming datasets...

             DATA PREPROCESSING QUALITY SCORECARD
 -> Final Processed Train Shape     : (533, 71)
 -> Final Processed Test Shape      : (134, 71)
 -> Total Expanded Sparse Features  : 71
 -> Remaining Null Values Vector    : 0
 -> Feature Mean Range Verification : Close to -0.00
 -> Feature Std Dev Verification    : 1.00
------------------------------------------------------------

First 3 Processed Samples Matrix Snippet (View of Standardized Columns):
   Account length  Area code  Number vm